In [1]:
!pip install shapely

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.4 MB 1.4 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.4 MB 1.6 MB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 1.4 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.4 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.3 MB/s eta 0:00:00


In [5]:
pip install dill

Note: you may need to restart the kernel to use updated packages.


In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from shapely.geometry import Point, Polygon
import torch
from PIL import Image

In [2]:
model = YOLO(r'D:\CriticalAI\weights\Aisle\best.pt')

In [4]:
# WEBCAM
fence_points = np.array([[275, 220], [350, 220], [400, 450], [200, 450]], np.int32)
fence_polygon = Polygon(fence_points)

# Open Webcam (0 for default webcam, change to 1 if using an external webcam)
cap = cv2.VideoCapture(0)

# Get video properties
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = 30  # Manually set an FPS for the webcam

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  

    # Run YOLO detection
    results = model(frame)
    boxes = results[0].boxes.xyxy.cpu().numpy()

    # Draw Virtual Fence
    cv2.polylines(frame, [fence_points], isClosed=True, color=(255, 0, 0), thickness=3)

    # Process each detected human
    for box in boxes:
        x1, y1, x2, y2 = map(int, box[:4])  
        feet_x = (x1 + x2) // 2
        feet_y = y2  

        feet_point = Point(feet_x, feet_y)

        if feet_point.within(fence_polygon):
            print("🚨 ALERT: Feet inside the virtual fence!")
            cv2.putText(frame, "ALERT!", (feet_x, feet_y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            cv2.circle(frame, (feet_x, feet_y), 5, (0, 0, 255), -1)  
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  
        else:
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)  

    # Show the frame (press 'q' to exit)
    cv2.imshow("Webcam - Human Detection with Virtual Fence", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


0: 480x640 2 Humans, 155.9ms
Speed: 11.4ms preprocess, 155.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 89.9ms
Speed: 8.4ms preprocess, 89.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 70.9ms
Speed: 1.9ms preprocess, 70.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 70.5ms
Speed: 2.0ms preprocess, 70.5ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 68.4ms
Speed: 2.3ms preprocess, 68.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 65.9ms
Speed: 1.9ms preprocess, 65.9ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 67.9ms
Speed: 1.7ms preprocess, 67.9ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Human, 61.6ms
Speed: 1.8ms preprocess, 61.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 6

#### 